In [ ]:
"""
Project Title: Elevation-based NDVI Analysis
Author: James McLoughlin
Date: May 2026
Description: This script processes DEM and Landsat data to calculate zonal statistics for vegetation health across different elevation zones.
"""

In [3]:
# =============================================================================
# Library Imports
# =============================================================================
# Standard library imports
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # silences future development warninigs

# Geospatial and Data Science libraries
import earthaccess           # For NASA Earthdata access and authentication
import folium                # For interactive web map visualization
import geopandas as gpd      # For vector data manipulation
import numpy as np           # For numerical array operations
np.seterr(divide='ignore', invalid='ignore')  # silences mathetmatical warnings (e.g.divide by zero in indices like NDVI)

import pandas as pd          # For tabular data management and CSV export
from pathlib import Path     # For cross-platform file path handling
import rasterio as rio       # Core library for reading and writing raster datasets
import rasterio.merge
from rasterio import features
from rasterio.merge import merge
from rasterio.windows import from_bounds
from rasterstats import zonal_stats # For calculating statistics within polygons
import rioxarray             # Extension for xarray to handle geospatial coordinates
import shapely               # For manipulation and analysis of geometric objects
from shapely.geometry import shape
import xarray as xr          # For handling multi-dimensional arrays
from xrspatial import hillshade # For terrain visualization
import matplotlib.pyplot as plt # For generating static plots and figures



# =============================================================================
# User Inputs and configuration
# =============================================================================
# Define the root directory for all project data to maintain relative paths
base = Path("C:/Users/jj_mc/OneDrive/Documents/GitHub/EGM722-Assessment/Data")

# Configuration dictionaries controlling which analysis and visualizations are processed
# 1 = Enable, 0 = Disable
ANALYSIS_OPTIONS = {
    "ndvi": 1,
    "ndwi": 0,
    "ndsi": 0
}

DISPLAY_OPTIONS = {
    "true_colour": 1,
    "false_colour": 0
}

# Defines vertical intervals for elevation analysis (meters)
step = 500

print("1. User Inputs and selections accepted")



# =============================================================================
# Predefined dictionary
# =============================================================================
# Maps specific tasks to the required spectral colours
# Allows script to load only necessary data for selected tasks
TASK_BANDS = {
    "true_colour" : ["red", "green","blue"],
    "false_colour" : ["nir","red","green"],
    "ndvi": ["red", "nir"],
    "ndwi": ["nir","green"],
    "ndsi": ["swir1","green"]
}



# =============================================================================
# Helper Functions
# =============================================================================
def raster_dataframe():
    """
    Searches a presepcified directory and sub directories to catalog Landsat 8 and 9 satellite imagery.
    
    This function iterates through specified data path, extracts image specific infotmation from 
    the Landsat filenames and maps bands the band number to the relative spectral color. It compiles 
    the information into a Pandas dataframe.

    Returns:
        pd.DataFrame: A DataFrame containing columns for filename, satellite ID, 
                      acquisition year, band ID, spectral colour, and file path.
    """
    records = []
    
    # a. Mapping Landsat 8/9 band identifiers to spectral colours 
    l8_9_bands = {
        "B1":"coast/aero", "B2":"blue", "B3":"green", "B4":"red", "B5":"nir", "B6":"swir1", "B7":"swir2"
    }
    
    # b. Iterates through folders within the configured Landsat data directory
    for folder in PATHS["landsat_data"].iterdir():
        if folder.is_dir():
            # Target Surface Reflectance (SR) TIF files
            for file in folder.glob("*_SR_B*.TIF"):
                parts = file.name.split("_")
                sat_id = parts[0][:4]  # Extract satellite ID, e.g. 'LC08'
                band_id = parts[-1].replace(".TIF", "")  # Extract band number, e.g. 'B1'

                # Assigns spectral color based on the satellite band mapping
                if sat_id in ["LC08", "LC09"]:
                    colour = l8_9_bands.get(band_id)

                # Only includes the files if the colour is required forthe study
                if colour in ALL_COLOURS:
                    records.append({
                        "filename": file.name,
                        "satellite": sat_id,
                        "year": parts[3][:4],
                        "band": band_id,
                        "colour": colour,
                        "path": str(file)
                    })
    
    # c. Return the collected metadata as a pandas DataFrame for analysis
    return pd.DataFrame(records)


    

def create_mosaic(file_list, out_path, dtype=None, nodata_val=None):
    """
    Merges multiple files into a single mosaicked image.

    Args:
        file_list (list): List of file paths (Path objects or strings) of the images to be merged.
        out_path (Path): Destination path for the generated mosaic (in .TIF formate).
        dtype (str, optional): Desired data type for output. Defaults to source dtype.
        nodata_val (float, optional): Value representing 'no data'. Defaults to source value.

    Returns:
        Path: The file path where the mosaic was successfully saved.
    """
    # a. Access the first file to establish baseline spatial metadata (CRS and Profile)    
    with rio.open(file_list[0]) as src:
        target_crs = src.crs
        out_meta = src.profile.copy()
        if nodata_val is None:
            nodata_val = src.nodata
        
    # b. Create the list of files to be merge.
    to_merge = []
    for f in file_list:
        with rio.open(f) as src:
                to_merge.append(f)
               
    # c. Mosiack the files, returns the combined array and new geotransform
    mosaic, out_trans = merge(to_merge, nodata=0)

    # d. File clean-up
    for s in to_merge:
        if hasattr(s, 'close'):
            s.close()
   
    # e. Synchronize metadata with the new mosaic dimensions and transformation
    bands, height, width = mosaic.shape
    out_meta.update({
        "height": height, 
        "width": width, 
        "transform": out_trans, 
        "nodata": 0,
        "dtype": dtype or out_meta['dtype']
    })

    # f. Write the array to disk
    with rio.open(out_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    print(f"     Mosaic saved: {out_path.name}")
    return out_path   # path used to build the dataset used for analysis 



# =============================================================================
# Main Worflow
# =============================================================================

# --- 2. Create folder structure ---
# =============================================================================
# Define project directory strcutre using pathlib
PATHS = {
    "landsat_data" : base / "Landsat_Rasters",
    "mosaics": base / "Mosaics",
    "vectors": base / "Vector_Layers",
    "earthaccess": base / "EarthAccess",
    "results": base / "Results"
}

# Batch directory creation, and checks parent exisits to ensure full path creation 
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)
print("2. Directories checked/created")



# --- 3. Determine colours bands required for selected tasks --- 
# =============================================================================
# Dictionary mapping selected analysis/outputs to their required spectral colours
TASK_BANDS = {
    "true_colour" : ["red", "green","blue"],
    "false_colour" : ["nir","red","green"],
    "ndvi": ["red", "nir"],
    "ndwi": ["nir","green"],
    "ndsi": ["swir1","green"]
}

# Identify active tasks by filtering through user-defined configuration  (where value == 1)
ANALYSIS_TASKS = [k for k, v in ANALYSIS_OPTIONS.items() if v == 1]
DISPLAY_TASKS = [k for k, v in DISPLAY_OPTIONS.items() if v == 1]

# Identify unique bands required across for analysis / display tasks, sets automatically handle de-duplication.
ANALYSIS_COLOURS = {band for task in ANALYSIS_TASKS for band in TASK_BANDS[task]}
DISPLAY_COLOURS = {band for task in DISPLAY_TASKS for band in TASK_BANDS[task]}

# Combine analysis and display colours into a master of unique colours using set union (|)
ALL_COLOURS = list(DISPLAY_COLOURS | ANALYSIS_COLOURS)
print("3. Band identification and mapping completed")



# --- 4. Build DataFrame of rasters needed for mosaicking ---
# =============================================================================
# Generate a Pandas dataframe of available local imagery matching the required bands
raster_df = raster_dataframe()

# Extract the year of the data being analysed
current_year = raster_df['year'].unique()[0]

# Export the dataframe to CSV
raster_df.to_csv(PATHS["landsat_data"] / f"{current_year}_raster_dataframe.csv", index=False)
print(F"4. Image dataset created : {len(raster_df)} files included")



# --- 5. Mosaick Rasters --- 
# =============================================================================
file_map = {}
print("5. Mosaicking started...")

# Group the dataframe by color to process all relevant tiles together
for colour, group in raster_df.groupby('colour'):
    # Extract file paths into a list for mosaicking
    file_list = group['path'].tolist()     
    
    # Define the output destination of the mosiac
    dst_path = PATHS["mosaics"] / f"{current_year}_{colour}_mosaic.tif"
    
    # Call mosaic function to merge tiles listed in file_lits
    # Returns the path of the mosaicked image, to populate file_map for use in Section 6
    mosaic_path = create_mosaic(file_list, dst_path, nodata_val=0)     
    
    # Store file path in a dictionary for easy access during Dataset assembly
    file_map[colour.lower()] = mosaic_path 

print("     All rasters mosaicked")



# --- 6. Build Dataset of Mosaicked Images ---
# =============================================================================
mosaic_list = []

# Iterate through the file-map established in the previous section
for name, path in file_map.items():
    
    # Load rasters using rioxarray with Dask-backed chunking.
    # Uses 'Lazy Loading' to process pieces of the raster as needed and prevent memory saturation.
    mosaic_da = rioxarray.open_rasterio(path, chunks={'x': 2048, 'y': 2048})
    
    # Pre-process DataArray for Dataset integration:
    # 1. .squeeze() removes the single 'band' dimension, resulting in a 2D array.
    # 2. .drop_vars() removes non-spatial coordinates that may conflict during merging.
    # 3. .rename() assigns the spectral band name (e.g., 'red') to the variable.
    mosaic_da = mosaic_da.squeeze().drop_vars('band', errors='ignore').rename(name)
    mosaic_list.append(mosaic_da)

# Combine individual arrays into a single multi-dimensional xarray Dataset
ds = xr.merge(mosaic_list)

print(f"6. Dataset for {current_year} completed")


    
# --- 7. Create Composite Images ---
# =============================================================================
print("7. Creating composite images")

# Iterate through each active display task (e.g., 'true_colour', 'false_colour')
for disp_tsk in DISPLAY_TASKS:
    
    # Slice the multi-variable Dataset to specific bands required for the task
    # .to_array() converts the specific bands raters into a multi-band composite
    img = ds[TASK_BANDS[disp_tsk]].to_array(dim='band')
    
    # Define destination for the hi-res composite array and save it.
    # tiled = True enable file to be opened quicker, compress="deflate" shrinks image file on disk 
    img_dest_hi = PATHS["results"] / f"{current_year}_{disp_tsk}_composite_hi_res.tif"
    img.rio.to_raster(img_dest_hi)#, tiled=True, compress="deflate")

    # Create and save low-res version for easier viewing / manipulation. 
    # Normalise the original composite from uint16 by converting to float32 and divdising by 
    # Reduce resolution by a factor of 100, .mean() aveages resampling, type = uint8 futher reduces memory demands


    # img_low_res = img.astype("uint8")
    # img_low_res = img_low_res.coarsen(x=10, y=10, boundary="trim").mean().astype("uint8")
    # img_dest_low = PATHS["results"] / f"{current_year}_{disp_tsk}_composite_low_res.tif"
    # img_low_res.rio.to_raster(img_dest_low)
    img_converted = (img.astype("float32")/65535.0)
    img_low_res = (img_converted.coarsen(x=10, y=10, boundary="trim").mean() * 255).astype("uint8")
    img_dest_low = PATHS["results"] / f"{current_year}_{disp_tsk}_composite_low_res.tif"
    img_low_res.rio.to_raster(img_dest_low)
    print(f"     Composites saved: {current_year} {disp_tsk}")

        

# # --- 8. NDVI and NDWI Analysis ---
# # =============================================================================
# # Tmo minmise memory usage reduce the rasters to the region of interst
# # Locate and read the Region of Interest (ROI) vector file
# roi = PATHS["vectors"] / "national_park_boundary.shp"
# roi_gdf = gpd.read_file(roi)

# # Reproject the ROI to match the dataset CRS
# roi_gdf = roi_gdf.to_crs(ds.rio.crs)

# # Clip the dataset to the ROI boundary and remove empty 'bounding box' pixels outside the polygon (drop=True)
# ds_clipped = ds.rio.clip(roi_gdf.geometry, ds.rio.crs, drop=True) 

# # Mask out zero values to avoid divisde by zero errors during calculation
# ds_clean = ds_clipped.where(ds_clipped > 0)

# # Iterate through active analysis tasks (e.g., NDVI, NDWI)
# for task in ANALYSIS_TASKS: 
#     # Retrieve required bands for the current index calculation
#     b1_name = TASK_BANDS[task][0] 
#     b2_name = TASK_BANDS[task][1] 

#     # Concert to float to avoid calaucation errors (Landsat 8/9 repreaented as uint16)
#     b1 = ds_clean[b1_name].astype(float)
#     b2 = ds_clean[b2_name].astype(float)

#     # Calaualted the Normalised Differnce Index
#     ds_clipped[task] = (b2-b1)/(b2+b1)

#     # Remove and points that have inf value (i.e. were /0)
#     ds_clipped[task] = ds_clipped[task].where(np.isfinite(ds_clipped[task])) 
# print("8. Normalized Difference Index Analysis Completed")



# # --- 9. Acquire DEMs from Earth Access---
# # =============================================================================
# # DEMs required for elevation based analysis. ROI polygon used to spatially slect DEMs from NASA EarthAceess
# # Reproject the ROI polygon to CRS requied for EarthAccess (EPSG:4326)
# roi_4326 = roi_gdf.to_crs(epsg = 4326).union_all()

# # Create mimnimum rotated retangle around ROI polygon (siple polygon needed for EarthAcress data search)
# search_area = roi_4326.minimum_rotated_rectangle

# # Change polygon vertices definiton to counter-clockwise orientation required by EarthAcess
# search_area = shapely.geometry.polygon.orient(search_area, sign=1) 

# print("9. EarthAccess data acquistion started...")

# # Log into EarthAccess using credentials in local .netric file
# earthaccess.login(strategy='netrc') 

# # Searh the ASTER Global Digital Elevation Model (ASTGTM) collection
# # Filter results using exterior coordinates of ROI's mimnimum rotated retangle
# earthaccess_files = earthaccess.search_data( 
#     short_name = 'ASTGTM',
#     polygon = search_area.exterior.coords
# )

# # Execute batch download to the local pre-defined directory
# earthaccess_dest = PATHS["earthaccess"]
# downloaded_files = earthaccess.download(earthaccess_files, earthaccess_dest)



# # --- 10. Mosaicking the DEMs ---
# # =============================================================================
# print("10. Mosaicking DEMs...")

# # Filter the downloaded files to target only the Digital Elevation Model TIFs
# dem_files = [fn for fn in downloaded_files if 'dem.tif' in fn.name]

# # Define destination for the mosaikced DEM 
# dem_mosaic_path = PATHS["mosaics"] / "DEM_mosaic.TIF" 

# # Call 'create_mosaic' helper fuction to merge and save mosaic. 
# create_mosaic(dem_files, dem_mosaic_path, dtype='float32')  #dtype set to 'float32' to minimise memory usage 

# print("     DEMs mosaicked")



# # --- 11. Vectorising Mosaicked DEM ---
# # =============================================================================
# # Extracting polygons from the DEM describing zones with different elevations
# print("11. Vectorising DEM...")

# # Load the DEM mosaic and remove the single band dimension
# dem_master = rioxarray.open_rasterio(dem_mosaic_path).squeeze() 

# # Align the ROI coordinate system to match the DEM
# roi_dem = roi_gdf.to_crs(dem_master.rio.crs)

# # Clip the DEM to the ROI to minimise memory usage during vectorization
# # Use drop=True to remove data outside the ROI
# dem_clipped = dem_master.rio.clip(roi_dem.geometry, drop=True)

# # Segment the elevation data into discrete elevation zones
# # Foor division (//) bins data into 'step' intervals
# # dtype "int32" used to minimise memory usage 
# dem_segmented = (dem_clipped.values // step).astype("int32")

# # Create a boolean mask to ensure only valid (non-NaN) data is vectorized
# mask = ~np.isnan(dem_segmented)

# # Extract polygons from the segmented raster corresponding to regions with the same elevtion zone
# shapes = features.shapes(dem_segmented, mask=mask,transform=dem_clipped.rio.transform())

# polygons = [] 

# # Loop through each polygon (geom) and elevation step (value)
# for i, (geom, value) in enumerate(shapes): 
    
#     #Scale polygon value back to actual elevation
#     zone_label = int(value*step) 
    
#     # Create a dictionary for each polygon with a unique ID, its elevation, and geometry
#     polygons.append({
#         'poly_id':i,
#         'elevation_zone': zone_label,
#         'geometry':shape(geom)
#     })

# # Convert list of dictionaries into GeoPandas GeoDataFrame and canage projection to that of oiginal DEM
# segments_gdf = gpd.GeoDataFrame(polygons, crs=dem_clipped.rio.crs)

# # Define output path and save the vector polygons as a .gpkg
# output_path = PATHS["vectors"] / "elevation_zones.gpkg"
# segments_gdf.to_file(output_path, driver="GPKG")

# print("      Vectorising completed and saved")



# # --- 12. Spatial Analysis ---
# # =============================================================================
# # Use the vetorisied polygons with the NDVI layers to obtain vegetation health statistics for different elevations 
# print("12. Calculating Zonal Statistics")

# # Reproject GeoDataFramce of vectorized elevation zones to match Landsat data CRS
# segments_gdf = segments_gdf.to_crs(ds_clipped.rio.crs)

# # Isolate NDVI data from dataset for statistical analysis
# ndvi_layer = ds_clipped.ndvi

# # Perform zonal statistics to calculate NDVI statistics by elevation zone
# stats = zonal_stats(
#     segments_gdf,                       # Vector polygons
#     ndvi_layer.values,                  # NDVI raster values
#     affine=ds_clipped.rio.transform(),  # Spatial orientation for the calculation
#     stats=['mean', 'std'],              # Calculation of mean and standard deviation
#     nodata=np.nan,                      # Ignore "empty" (NaN) pixels
#     all_touched=True                    # Include pixels partly covered by polygons
# )

# # Combine results into a list of dictionaries
# all_results = []
# for i, s in enumerate(stats):
#     all_results.append({
#         'year': current_year,
#         'height': segments_gdf.iloc[i]['elevation_zone'],  # Polygon height value
#         'mean': s['mean'],                                 # Average NDVI for polygon
#         'std': s['std']                                    # NDVI std over polygon
#     })

# # Convert the list into a Pandas DataFrame for exporting and plotting
# full_stats_df = pd.DataFrame(all_results)

# # Define output path and save the statistical analysis to CSV
# output_file = PATHS["results"] / f"final_stats_{current_year}.csv" # file path for zonal statistics data
# full_stats_df.to_csv(output_file, index=False) # Save statistics as CSV

# print("      Zonal statistics calculated.")
print("--- Code completed ---")

1. User Inputs and selections accepted
2. Directories checked/created
3. Band identification and mapping completed
4. Image dataset created : 8 files included
5. Mosaicking started...
     Mosaic saved: 2025_blue_mosaic.tif
     Mosaic saved: 2025_green_mosaic.tif
     Mosaic saved: 2025_nir_mosaic.tif
     Mosaic saved: 2025_red_mosaic.tif
     All rasters mosaicked
6. Dataset for 2025 completed
7. Creating composite images
     Composites saved: 2025 true_colour
--- Code completed ---


In [ ]:
img.max()

In [ ]:
import rioxarray
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# Define project directory strcutre using pathlib
PATHS = {
    "mosaics": base / "Mosaics",
    "vectors": base / "Vector_Layers",
    "results": base / "Results"
}

# 1. Open the image
# 'mask_and_scale=True' automatically converts nodata to NaN
ds = rioxarray.open_rasterio(PATHS['results']/"2025_true_colour_composite_low_res.tif", mask_and_scale=True)

# 2. Color Balance (Calculate quantiles across the whole spatial extent)
vmin = ds.quantile(0.02)
vmax = ds.quantile(0.98)

# 3. Scale and Apply Gamma
gamma = 0.8
img_scaled = ((ds - vmin) / (vmax - vmin)).clip(0, 1)
img_scaled = np.power(img_scaled, gamma)

# 4. Plot
fig, ax = plt.subplots(figsize=(15, 15))

# Xarray's .plot.imshow handles the coordinates and RGB conversion automatically
img_low_res.plot.imshow(ax=ax)

# 5. Overlay Polygons (matches the CRS automatically from the DataArray)
segments_gdf.to_crs(ds.rio.crs).plot(ax=ax, facecolor ='none',edgecolor ='lightblue',linewidth=0.5)
roi_gdf.to_crs(ds.rio.crs).plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)

plt.show()

In [ ]:
import rasterio
from rasterio.plot import show
import geopandas as gpd
import matplotlib.pyplot as plt

gamma = 0.8
for year in df["year"].unique():
    for disp_tsk in DISPLAY_TASKS:
        img = ds[INDEX_TO_BANDS[disp_tsk]].sel(year=year).to_array(dim='band')
        #vimn, vmax = img.min(), img.max()
        #vmin, vmax = img.quantile([0.02, 0.98]).compute()
        vmin, vmax = img.where(img > 0).quantile([0.05, 0.99]).compute()
        img_scaled = ((img - vmin)/(vmax-vmin)*255).clip(0, 255)
        img_scaled = (255 * (img_scaled / 255) ** gamma).astype("uint8")
        img_low_res = img_scaled.coarsen(x=10, y=10, boundary ="trim").mean().astype("uint8")
        img_dest = PATHS["results"] / f"{year}_{disp_tsk}_composite.tif"
        img_low_res.rio.to_raster(img_dest)
        print(f"Saved: {img_dest.name}") 

        
landsat_image = PATHS['results']/"2025_true_colour_composite.tif"
polygons = PATHS['results']/"final_vegetation_stats.csv"
# 1. Open the Landsat raster
with rasterio.open(landsat_image) as src:
    fig, ax = plt.subplots(figsize=(15, 15))
    
    # 2. Display the raster (e.g., using the first three bands for RGB)
    # Use transform=src.transform to keep the plot in map coordinates
    show(src, ax=ax, title="Landsat Overlay")

    # 3. Load and reproject your polygon dataframe
    roi_4326 = roi_gdf.to_crs(epsg = 26912)
#    gdf = gpd.read_file(polygons)
#    gdf = gdf.to_crs(src.crs) # Vital for alignment

    # 4. Overlay the polygons
    segments_gdf.plot(ax=ax, facecolor ='none',edgecolor ='lightblue',linewidth=0.5)
    roi_4326.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)

plt.show()

In [ ]:
for yr in veg_stats_df['year'].unique():
    # Filter the data for just this year
    yr_data = veg_stats_df[veg_stats_df['year'] == yr].sort_values('height')
    
    plt.plot(
        yr_data['height'], 
        yr_data['mean'], 
        marker='o', 
        label=yr
    )

plt.title("Vegetation Health (NDVI) by Elevation")
plt.xlabel("Elevation Base (meters)")
plt.ylabel("Mean NDVI")
plt.legend(title="Year")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
## backup hillsdahes that worked


print("8. DEM mosaicking started...")
dem_files = [fn for fn in downloaded_files if 'dem.tif' in fn.name]
dem_mosaic_dest = PATHS["earthaccess"] / "DEM_mosaic.TIF"
dem_meta = create_mosaic(dem_files, dem_mosaic_dest, dtype='float32')
dem_ds = xr.Dataset({"elevation":dem_windowed})
dem_ds = dem_ds.rio.write_crs(win_meta['crs']).rio.write_transform(win_meta['transform'])
dem_ds_clipped = dem_ds.rio.clip(border_gdf_proj.geometry, drop =True)
hillshade_orig = hillshade(dem_ds, azimuth=225, angle_altitude=45)
hillshade_26912 = hillshade_orig.rio.reproject("EPSG:26912")
hillshade_dest = PATHS["results"] / "hillshade.tif"
hillshade_26912.rio.to_raster(hillshade_dest, compress = "lzw")

# --- 7. Creating Hillshde
hillshade = hillshade(dem_ds, azimuth=335, angle_altitude=45)
#hillshade_26912 = hillshade_orig.rio.reproject("EPSG:26912")
hillshade_dest = PATHS["results"] / "Hillshade.tif"
#hillshade_26912.rio.to_raster(hillshade_dest, compress = "lzw")

In [ ]:
gamma = 0.8
for year in df["year"].unique():
    for disp_tsk in DISPLAY_TASKS:
        img = ds[INDEX_TO_BANDS[disp_tsk]].sel(year=year).to_array(dim='band')
        #vimn, vmax = img.min(), img.max()
        #vmin, vmax = img.quantile([0.02, 0.98]).compute()
        vmin, vmax = img.where(img > 0).quantile([0.05, 0.99]).compute()
        img_scaled = ((img - vmin)/(vmax-vmin)*255).clip(0, 255)
        img_scaled = (255 * (img_scaled / 255) ** gamma).astype("uint8")
        img_low_res = img_scaled.coarsen(x=10, y=10, boundary ="trim").mean().astype("uint8")
        img_dest = PATHS["results"] / f"{year}_{disp_tsk}_composite.tif"
        img_low_res.rio.to_raster(img_dest)
        print(f"Saved: {img_dest.name}")    


scale = 2
img = rioxarray.open_rasterio(PATHS["results"]/"2020_true_colour_composite.tif")
img_height_new = int(img.rio.height / scale)
img_width_new = int(img.rio.width / scale)
img_scaled = img.rio.reproject('EPSG:4326', shape=(img_height_new,img_width_new))
b = img_scaled.rio.bounds()
img_bounds_folium = [[b[1],b[0]],[b[3],b[2]]]
img_data = img_scaled.transpose('y','x','band').values
#if img_data.max() <= 1.0:
#    img_data = (img_data * 255).astype('uint8')
alpha = np.where(img_data.sum(axis=-1) > 0, 255, 0).astype('uint8')
img_data_rgba = np.dstack((img_data, alpha))

hill = rioxarray.open_rasterio(PATHS["results"]/"terrain_hillshade_high_res.tif")
hill_height_new = int(hill.rio.width / scale)
hill_width_new = int(hill.rio.width / scale)
hill_scaled = hill.rio.reproject('EPSG:4326', shape=(hill_height_new,hill_width_new))
h = hill_scaled.rio.bounds()
hill_bounds_folium = [[h[1],h[0]],[h[3],h[2]]]
hill_data_2d = hill_scaled.values.squeeze()
#beta = np.where(hill_data.sum(axis=-1) > 0, 255, 0).astype('uint8')
#hill_data_2d = np.dstack((hill_data, beta))



m = folium.Map([36.15, -112.7],tiles="USGS.USImagery")
folium.raster_layers.ImageOverlay(
    name = "True Colour Overaly",
    image = img_data_rgba,
    bounds = img_bounds_folium,
).add_to(m)
folium.raster_layers.ImageOverlay(
    name = "Hillshade Overaly",
    image = hill_data_2d,
    bounds = hill_bounds_folium,
    opacity = 0.4    
).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Create a mask for the river
river_mask = ds_clipped['ndwi'].isel(year=0) > 0.1
# Apply this to NDVI to analyze only 'Land' pixels
ndvi_land_only = ds_clipped['ndvi'].where(~river_mask)

def mask_from_raster(year, pathrow, NDI):#, river_ref_path):
    poly_list = []
    file = NDI_data/f"{year}_{pathrow}_{NDI}.tif"
    with rio.open(file) as src:
        non_binary = src.read(1)
        mask = np.where(non_binary > -0.1 , 255 ,0).astype("uint8")
        cleaned_mask = sieve(mask,50)
        out_meta = src.meta.copy()
        out_meta.update(dtype = "uint8", nodata = 0)
        mask_out = masks / f"{year}_{pathrow}_{NDI}.tif"
        
        with rio.open(mask_out, "w", **out_meta) as dst:
            dst.write(cleaned_mask,1)

    file = mask_data / f"{year}_{pathrow}_{NDI}.tif"
    with rio.open(file) as src:
        tile = src.read(1).astype("int16")
        out_meta = src.meta.copy()
        mask_shapes = features.shapes(
            tile,
            mask = tile > 0,
            transform = out_meta['transform']
        )
    # The corrected list comprehension
    polygons = [
    {'geometry':shape(s),'properties':{'id': i}}
     for i, (s,v) in enumerate(mask_shapes)
    ]
              
    gdf = gpd.GeoDataFrame.from_features(polygons, crs=out_meta['crs'])
    return gdf

    